In [ ]:
import sys

In [ ]:
# sys.path.append("/home/gputnam/Diffusion-Anomaly-Detection/diffusion-anomaly")
sys.path.append("anomaly-detection/diffusion-anomaly")

In [ ]:
from guided_diffusion.script_util import (
    model_and_diffusion_defaults,
    diffusion_defaults,
    create_model_and_diffusion,
    args_to_dict,
    add_dict_to_argparser,
    create_gaussian_diffusion
)

from guided_diffusion.resample import UniformSampler
from guided_diffusion import dist_util

from guided_diffusion.fp16_util import *

from guided_diffusion.respace import space_timesteps

import numpy as np
import matplotlib.pyplot as plt
import torch as th

import h5py

import os

import pickle

In [ ]:
plt.rcParams.update({'font.size': 14})

In [ ]:
# imgs = np.expand_dims(np.stack([undershoot, bad_wire, dead_wires]), axis=1)
# imgs = th.tensor(imgs, device=dist_util.dev())

In [ ]:
def visualize(img):
    return np.clip(img, -1, 1)

In [ ]:
# Downsample this_arr by a factor of 4 (average over 4x4 blocks)
def downsample_by(arr, factor):
    h, w = arr.shape
    # Make sure dimensions are multiples of 4
    h_new = (h // factor) * factor
    arr_cropped = arr[:h_new, :]
    # Reshape and mean
    arr_down = arr_cropped.reshape(h_new//factor, factor, w, 1).mean(axis=(1, 3))
    return arr_down

In [ ]:
# # diagonal noise event
# filename = "/exp/sbnd/data/users/munjung/anomaly-detection/file35.h5"
# frames = h5py.File(filename, "r")
# data = frames.get("{}/raw".format(6))

# shadow event
filename = "/exp/sbnd/data/users/munjung/anomaly-detection/file50.h5"
frames = h5py.File(filename, "r")
data = frames.get("{}/raw".format(6))

data = np.array(data)
data = data.T

this_arr = data[:, 1984*2:]/100.

plt.imshow(this_arr, vmin=-0.2, vmax=0.2, cmap="bwr", aspect="auto")
plt.colorbar()

In [ ]:
allarrs = []
nrows, ncols = (512, 512)
planeno = 2
planearr = data[:, 4250:]
# planearr = data[:, 3968:5638]
h, w = planearr.shape
planearr_cropped = planearr[:(h // nrows) * nrows, :(w // nrows) * nrows]
H_crop, W_crop = planearr_cropped.shape
n_patches_h = H_crop // nrows
n_patches_w = W_crop // ncols
ll = planearr_cropped.reshape(n_patches_h, nrows, n_patches_w, ncols).swapaxes(1, 2).reshape(
    -1, nrows, ncols
)

# Use with stitch_patches(...) after inference to rebuild the cropped plane (H_crop x W_crop).
patch_layout = {
    "patch_size": (nrows, ncols),
    "grid_shape": (n_patches_h, n_patches_w),
    "cropped_shape": (H_crop, W_crop),
    "full_plane_shape": (h, w),
}

cscale = [200., 100., 200.][planeno]
allarrs.append(ll / cscale)

imgs = visualize(np.expand_dims(np.concatenate(allarrs), axis=1)).astype(np.float32)

In [ ]:
def stitch_patches(patches, grid_shape, patch_size=None):
    """Rebuild a 2D plane from row-major patches (same order as cell above).

    Parameters
    ----------
    patches : np.ndarray or th.Tensor
        Shape (N, ph, pw) or (N, C, ph, pw) with N = grid_shape[0] * grid_shape[1].
    grid_shape : tuple[int, int]
        (n_patches_h, n_patches_w); pass patch_layout['grid_shape'].
    patch_size : tuple[int, int] | None
        (ph, pw); defaults to patches.shape[-2:].

    Returns
    -------
    Stitched array/tensor: (H, W) or (C, H, W) matching patch_layout['cropped_shape'].
    """
    nh, nw = grid_shape
    if patch_size is None:
        ph, pw = int(patches.shape[-2]), int(patches.shape[-1])
    else:
        ph, pw = patch_size

    if isinstance(patches, th.Tensor):
        if patches.dim() == 4:
            n, c, _, _ = patches.shape
            x = patches.view(nh, nw, c, ph, pw).permute(2, 0, 3, 1, 4).reshape(c, nh * ph, nw * pw)
            return x
        if patches.dim() == 3:
            x = patches.view(nh, nw, ph, pw).permute(0, 2, 1, 3).reshape(nh * ph, nw * pw)
            return x
        raise ValueError(f"Expected 3D or 4D tensor, got {patches.dim()}D")

    if patches.ndim == 4:
        n, c, _, _ = patches.shape
        return patches.reshape(nh, nw, c, ph, pw).transpose(0, 2, 1, 3, 4).reshape(c, nh * ph, nw * pw)
    if patches.ndim == 3:
        return patches.reshape(nh, nw, ph, pw).swapaxes(1, 2).reshape(nh * ph, nw * pw)
    raise ValueError(f"Expected 3D or 4D array, got {patches.ndim}D")



In [ ]:
data.shape
imgs.shape

In [ ]:
for i in range(12):
    plt.figure(i)
    plt.imshow(np.squeeze(imgs[i]), vmin=-0.5, vmax=0.5)

In [ ]:
imgs = th.tensor(imgs, device=dist_util.dev())
th.set_grad_enabled(False)

In [ ]:
args = model_and_diffusion_defaults()
diffusion_args = diffusion_defaults()

In [ ]:
# MODEL
args["image_size"] = 512
args["num_channels"] = 32
args["class_cond"] = False
args["num_res_blocks"] = 2
args["num_heads"] = 8
args["learn_sigma"] = True
args["use_scale_shift_norm"] = False
args["attention_resolutions"] = "16,32"
args["channel_mult"] = "1,2,4,8,8,8"

# DIFFUSION
diffusion_args["diffusion_steps"] = 1000
diffusion_args["noise_schedule"] = "linear"
diffusion_args["rescale_learned_sigmas"] = False
diffusion_args["rescale_timesteps"] = False

diffusion_args.pop("diffusion_steps")
diffusion_args.pop("timestep_respacing")

# TODO: change?
diffusion_args["learn_sigma"] = True

args = args | diffusion_args

In [ ]:
model, diffusion = create_model_and_diffusion(**args)

In [ ]:
MODEL = "/exp/sbnd/data/users/gputnam/training-SBND/iterE/results/brats2update111000.pt"

In [ ]:
model.load_state_dict(
    dist_util.load_state_dict(MODEL, map_location="cpu")
)

model.to(dist_util.dev())
_ = model.eval()

In [ ]:
# # diagonal noise event
# T = 200
# x_lo, x_hi = 520, 1000
# y_lo, y_hi = 500, 2000

# shadow event
T = 150
x_lo, x_hi = 300, 400
y_lo, y_hi = 0, 3500


In [ ]:
ddim_noisef = diffusion.ddim_sample_loop_progressive(model, imgs.shape, time=T, noise=imgs, 
                                             reverse=True, progress=True)

ddim_noise = list(ddim_noisef)

In [ ]:
ddim_noised = ddim_noise[-1]["sample"]

In [ ]:
rand_noised = diffusion.q_sample(imgs, th.tensor(T, device=dist_util.dev()))

In [ ]:
ddim_2_ddim = diffusion.ddim_sample_loop_progressive(model, 
                imgs.shape, time=T, noise=ddim_noised, progress=True)

rand_2_ddim = diffusion.ddim_sample_loop_progressive(model, 
                imgs.shape, time=T, noise=rand_noised, progress=True)

ddim_2_ddpm = diffusion.p_sample_loop_progressive(model, 
                imgs.shape, time=T, noise=ddim_noised, progress=True)

rand_2_ddpm = diffusion.p_sample_loop_progressive(model, 
                imgs.shape, time=T, noise=rand_noised, progress=True)

In [ ]:
ddim_2_ddim_reco = list(ddim_2_ddim)[-1]["sample"]
rand_2_ddim_reco = list(rand_2_ddim)[-1]["sample"]
# ddim_2_ddpm_reco = list(ddim_2_ddpm)[-1]["sample"]
rand_2_ddpm_reco = list(rand_2_ddpm)[-1]["sample"]

In [ ]:
original = stitch_patches(imgs.cpu().squeeze(1).numpy(), patch_layout["grid_shape"])
full_reco = stitch_patches(ddim_2_ddim_reco.cpu().squeeze(1).numpy(), patch_layout["grid_shape"])
sal_map = full_reco - original

reco_names = ["Original", "Reconstructed", "Saliency"]
reco_frames = [original, full_reco, sal_map]

fig, axs = plt.subplots(1, 3, figsize=(12,3))

for ireco, rname in enumerate(reco_names):
    if ireco == 2:
        axs[ireco].imshow(reco_frames[ireco][y_lo:y_hi, x_lo:x_hi], vmin=-0.1, vmax=0.1, aspect="auto", cmap="bwr")
        # axs[ireco].imshow(reco_frames[ireco], vmin=-0.1, vmax=0.1, aspect="auto", cmap="bwr")
    else:
        axs[ireco].imshow(reco_frames[ireco][y_lo:y_hi, x_lo:x_hi], vmin=-0.5, vmax=0.5, aspect="auto")
        # axs[ireco].imshow(reco_frames[ireco], vmin=-0.5, vmax=0.5, aspect="auto", cmap="bwr")

    axs[ireco].set_title(rname, fontsize=12)

    if ireco != 0:
        axs[ireco].set_yticks([])
        
fig.subplots_adjust(wspace=0.2, hspace=0.5) # Make room on the right

fig.text(0.05, 0.95, "SBND Data", horizontalalignment="left", 
                verticalalignment="bottom", fontsize=16, fontweight="bold")

fig.supxlabel("Wire Number", fontsize=12, y=-0.07)
fig.supylabel("Time Tick", fontsize=12, x=0.05)

# Inspect Individual Patches

In [ ]:
recos = [
    ddim_2_ddim_reco,
    rand_2_ddim_reco,
    # ddim_2_ddpm_reco,
    rand_2_ddpm_reco
]

In [ ]:
titles = [
    "Shower Undershoot",
    "Bad Wire",
    "Dead Wires",
]

reco_names = [
    "DDIM to DDIM",
    "Random to DDIM",
    # "DDIM to DDPM",
    "Random to DDPM",
]

In [ ]:
for ifig, title in enumerate(range(12)):
    plt.figure(ifig)
    fig, axs = plt.subplots(2, 4, figsize=(9.6,5))

    axs[0][0].set_title("Original Image")
    axs[0][0].imshow(np.squeeze(imgs[ifig].cpu().numpy()), vmin=-0.5, vmax=0.5)
    axs[1][0].axis('off')

    # axs[1][0].text(0.05, 0.5, title.replace(" ", "\n"), horizontalalignment="left", 
    #                verticalalignment="center", fontsize=18, fontweight="bold")
    
    for ireco, rname in enumerate(reco_names):
        axs[0][ireco+1].imshow(np.squeeze(recos[ireco][ifig].cpu().numpy()), vmin=-0.5, vmax=0.5)
        axs[1][ireco+1].imshow(np.squeeze((imgs[ifig] - recos[ireco][ifig]).cpu().numpy()), vmin=-0.1, vmax=0.1, cmap="bwr")
        
        axs[0][ireco+1].set_xticks([])
        axs[0][ireco+1].set_yticks([])
        
        if ireco > 0:
            axs[1][ireco+1].set_yticks([])
            
        axs[0][ireco+1].set_title(rname, fontsize=12)
            
    fig.subplots_adjust(wspace=0.02, hspace=0.2) # Make room on the right
    fig.suptitle("Reconstructed Images", x=0.62)
    axs[1][2].set_title("Sailiency Maps")
    
    axs[0][0].set_xlabel("Wire Number")
    axs[0][0].set_ylabel("Time Tick")